# Wavelet-YOLOv12 â€” Chen Split (Tuberculosis6208)

Inline training notebook â€” `model.train()` dan semua hyperparam terlihat langsung di cell.

**Setup:**
- Dataset zip di Drive: `MyDrive/Tuberculosis6208.zip` (Pascal-VOC format)
- Split: **Chen et al. IJAI 2024** â€” `1024 / 140 / 101`, `SPLIT_SEED=42` (deterministic)
- Default: **1 run** (model = wavelet full, seed = 42)
- Logging: **W&B** â€” project `wavelet_yolo12_chen`

**Runtime:** A100 â‰ˆ 25â€“30 menit untuk 1 run.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repo (branch `dev/wavelet`)

In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/wavelet-yolo12')
BRANCH   = 'dev/wavelet'

if REPO_DIR.exists():
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull --ff-only
else:
    !git clone -b {BRANCH} https://github.com/iswantosan/wavelet-yolo12.git {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('cwd:', os.getcwd())
!git log -1 --oneline

## 3. Install dependencies (editable, supaya `WaveDown` ke-load)

In [ ]:
!pip -q install -e . wandb

In [ ]:
import torch, ultralytics
from ultralytics.nn.modules import WaveDown, HaarDWT
print('torch       :', torch.__version__, '| cuda:', torch.cuda.is_available())
print('GPU         :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('ultralytics :', ultralytics.__version__)
print('WaveDown OK :', WaveDown is not None and HaarDWT is not None)

## 4. Build Chen split (1024 / 140 / 101, seed=42)

Extract zip â†’ konversi VOC XML â†’ YOLO `.txt` â†’ deterministic shuffle â†’ tulis `data.yaml`. Skip kalau output sudah ada.

In [ ]:
DRIVE_ZIP    = '/content/drive/MyDrive/Tuberculosis6208.zip'
EXTRACT_DIR  = '/content/dataset/raw'
RAW_DIR      = f'{EXTRACT_DIR}/Tuberculosis6208/tuberculosis-phonecamera'
SPLIT_DIR    = '/content/tb_chen_split'
DATA_YAML    = f'{SPLIT_DIR}/data.yaml'

!python scripts/build_chen_split.py \
    --zip "{DRIVE_ZIP}" --extract-dir "{EXTRACT_DIR}" \
    --src "{RAW_DIR}" --out "{SPLIT_DIR}"

!ls -la {SPLIT_DIR} && echo '---' && cat {DATA_YAML}

## 5. Smoke test (build model + dummy forward)

In [ ]:
!python scripts/smoke_test_wavelet.py

## 6. W&B login

Paste API key dari https://wandb.ai/authorize ketika di-prompt.

In [ ]:
import wandb
wandb.login()

## 7. Config run

Ganti `MODEL_CFG` ke salah satu (filename ber-suffix `s` → scale `s` auto-detected → ~9.1M params, match `yolov12s.pt` pretrained):
- `ultralytics/cfg/models/v12/yolov12s.yaml` — baseline (no wavelet)
- `ultralytics/cfg/models/v12/yolov12s-wavelet-p3.yaml` — WaveDown di P3 saja
- `ultralytics/cfg/models/v12/yolov12s-wavelet.yaml` — WaveDown di P3+P4+P5 (default)

In [ ]:
MODEL_CFG    = "ultralytics/cfg/models/v12/yolov12s-wavelet.yaml"   # scale s -> 9.1M params
PRETRAINED   = "yolov12s.pt"       # auto-download, matches scale
SEED         = 42
EPOCHS       = 60
IMGSZ        = 640
BATCH        = 16
DEVICE       = 0

WANDB_PROJECT = "wavelet_yolo12_chen"
RUN_PROJECT   = "/content/runs/wavelet_chen"
RUN_NAME      = f"{Path(MODEL_CFG).stem}_seed{SEED}_{EPOCHS}ep"

print("cfg     :", MODEL_CFG)
print("seed    :", SEED)
print("epochs  :", EPOCHS)
print("run_name:", RUN_NAME)

## 8. Seed + W&B init

In [ ]:
import os, gc, random, numpy as np, torch

# Stable SDP kernel (avoid flash/mem-efficient mismatch on Ampere/Ada)
os.environ['PYTORCH_SDP_KERNEL'] = 'math'
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

# Reproducibility
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Disable Ultralytics' built-in W&B callback â€” kita log manual
from ultralytics.utils import SETTINGS
SETTINGS.update({'wandb': False})

run = wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    reinit=True,
    config=dict(
        model_cfg=MODEL_CFG, data_yaml=DATA_YAML, pretrained=PRETRAINED,
        seed=SEED, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        optimizer='SGD', lr0=0.01, momentum=0.937, cos_lr=True,
        split='chen_1024_140_101', split_seed=42,
    ),
    tags=[Path(MODEL_CFG).stem, f'seed{SEED}', 'chen_split'],
)
print('W&B run:', run.url)

## 9. Train â€” `model.train()` eksplisit, semua hyperparam terlihat

In [ ]:
import time
from ultralytics import YOLO

model = YOLO(MODEL_CFG)
try:
    model.load(PRETRAINED)
    print(f'Loaded pretrained: {PRETRAINED}')
except Exception as e:
    print(f'[warn] could not load pretrained: {e}')

t0 = time.time()
results = model.train(
    # data + scale
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    # optimizer
    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    cos_lr=True,
    patience=0,
    # precision + reproducibility
    amp=True,
    deterministic=True,
    seed=SEED,
    workers=8,
    # augmentation (mirror thesis)
    hsv_h=0.1, hsv_s=0.3, hsv_v=0.3,
    degrees=10, translate=0.05, scale=0.3, shear=0.0, perspective=0.0,
    flipud=0.5, fliplr=0.5,
    mosaic=0.5, mixup=0.3, auto_augment=None,
    # output
    project=RUN_PROJECT,
    name=f'{RUN_NAME}_train',
    exist_ok=True,
    save=True,
    verbose=True,
)
train_secs = time.time() - t0
print(f'\nTrain time: {train_secs/60:.1f} min')
print(f'Save dir  : {results.save_dir}')

## 10. Log per-epoch curves ke W&B (dari `results.csv`)

In [ ]:
import pandas as pd

wandb.define_metric('epoch')
for k in [
    'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'train/total_loss',
    'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'val/total_loss',
    'val/mAP50', 'val/mAP50-95', 'val/precision', 'val/recall', 'lr/pg0',
]:
    wandb.define_metric(k, step_metric='epoch')

csv_path = Path(results.save_dir) / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path); df.columns = [c.strip() for c in df.columns]
    col_map = [
        ('train/box_loss', 'train/box_loss'),
        ('train/cls_loss', 'train/cls_loss'),
        ('train/dfl_loss', 'train/dfl_loss'),
        ('val/box_loss', 'val/box_loss'),
        ('val/cls_loss', 'val/cls_loss'),
        ('val/dfl_loss', 'val/dfl_loss'),
        ('metrics/mAP50(B)', 'val/mAP50'),
        ('metrics/mAP50-95(B)', 'val/mAP50-95'),
        ('metrics/precision(B)', 'val/precision'),
        ('metrics/recall(B)', 'val/recall'),
        ('lr/pg0', 'lr/pg0'),
    ]
    for _, row in df.iterrows():
        try: ep = int(row.get('epoch', 0))
        except Exception: continue
        log = {'epoch': ep}
        for src, dst in col_map:
            if src in df.columns:
                try: log[dst] = float(row[src])
                except Exception: pass
        tb, tc, td = log.get('train/box_loss'), log.get('train/cls_loss'), log.get('train/dfl_loss')
        if None not in (tb, tc, td): log['train/total_loss'] = tb + tc + td
        vb, vc, vd = log.get('val/box_loss'), log.get('val/cls_loss'), log.get('val/dfl_loss')
        if None not in (vb, vc, vd): log['val/total_loss'] = vb + vc + vd
        run.log(log)
    print('Logged per-epoch curves to W&B.')
else:
    print('results.csv not found at', csv_path)

## 11. Test eval di holdout 101-image split

In [ ]:
best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
print('Best ckpt:', best_pt)

eval_model = YOLO(str(best_pt))
eva = eval_model.val(data=DATA_YAML, split='test', imgsz=IMGSZ, device=DEVICE, verbose=False)

map50   = float(eva.box.map50)
map5095 = float(eva.box.map)
precision = float(np.mean(np.atleast_1d(eva.box.p)))
recall    = float(np.mean(np.atleast_1d(eva.box.r)))

# mAP@IoU=0.9 (index 8 of [0.5,0.55,...,0.95])
map_at_09 = float('nan')
try:
    ap_all = eva.box.all_ap
    if ap_all is not None and len(ap_all):
        ap = ap_all.mean(axis=0) if (hasattr(ap_all, 'ndim') and ap_all.ndim == 2) else ap_all
        if len(ap) >= 9: map_at_09 = float(ap[8])
except Exception as e:
    print(f'  (mAP@0.9 extract failed: {e})')

print(f'\n=== TEST RESULTS ({RUN_NAME}) ===')
print(f'  mAP50     : {map50:.4f}')
print(f'  mAP50-95  : {map5095:.4f}')
print(f'  mAP@0.9   : {map_at_09:.4f}')
print(f'  precision : {precision:.4f}')
print(f'  recall    : {recall:.4f}')
print(f'  train_min : {train_secs/60:.1f}')

run.summary['test/mAP50']     = map50
run.summary['test/mAP50-95']  = map5095
run.summary['test/mAP@0.9']   = map_at_09
run.summary['test/precision'] = precision
run.summary['test/recall']    = recall
run.summary['train/time_min'] = train_secs / 60

# Upload plots
for img in Path(results.save_dir).glob('*.png'):
    if any(t in img.stem.lower() for t in ('results', 'confusion', 'f1_curve', 'pr_curve', 'p_curve', 'r_curve')):
        try: run.log({f'plots/{img.stem}': wandb.Image(str(img))})
        except Exception: pass

run.finish()
print('\nW&B run finalised:', RUN_NAME)

## 12. (Opsional) Quick predict sample

In [ ]:
preds = eval_model.predict(
    source=f'{SPLIT_DIR}/test/images',
    save=True, imgsz=IMGSZ, conf=0.25, device=DEVICE,
)
print('Predictions saved to:', preds[0].save_dir if preds else None)